# R2R capacity on small images (held-out DIV2K crops): which adapted-parameter subset avoids fitting the noise?

Per-image R2R gain tuning of ADM gains +0.15 dB on Urban100 (about 2.4 M noisy values per image, 11× the 216 k
adapted parameters). It loses on Kodak24 (−0.02), McMaster (−0.09) and gray BSD68 / Set12 (−0.2 to −1.2), where
there are fewer noisy values per parameter. This run chooses a smaller subset on held-out data (amendment 8).

* **Data:** DIV2K valid 0801–0806 centre-cropped to 768×512 (Kodak-sized) and 0807–0812 to 512×512 (McMaster-sized),
  σ = 50, DMID's noise protocol.
* **Subsets:**
  * full: per-channel conv gains + GroupNorm γ/β (the reference);
  * norm: GroupNorm γ/β only;
  * decoder: per-channel gains in `output_blocks` and `out` only;
  * scalar: one scalar gain per convolution.
* **Fixed:** 90 steps, Muon (AdamW branch) at lr 1e-3, α = 0.5, 4 crops of 256². Scoring: whole image, one step,
  unadapted versus adapted.
* **Rule:** the subset with the highest mean gain, provided it is > 0 in both size buckets; otherwise none.

Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import io, json, math, os, queue, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
from PIL import Image

SIGMA, ALPHA, LR, CROP, BATCH, STEPS = 50, 0.5, 1e-3, 256, 4, 90
SIZES = {"768x512": (512, 768, range(801, 807)), "512x512": (512, 512, range(807, 813))}
SUBSETS = ["full", "norm", "decoder", "scalar"]
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__)

MUON_SRC = r'''"""Muon: momentum orthogonalised by Newton–Schulz, for hidden weight matrices.

Muon (Jordan et al., 2024) replaces the update of each weight matrix with the
nearest semi-orthogonal matrix to its Nesterov momentum, computed by five
quintic Newton–Schulz iterations; convolution kernels are flattened to
``(out, in · kh · kw)``. It is meant for hidden matrices only, so embeddings,
the output layer, biases and normalisation gains are updated by AdamW inside
the same optimizer, as in the reference implementation.

Muon updates are scaled by ``0.2 · sqrt(max(rows, columns))`` (Liu et al.,
2025, "Muon is Scalable for LLM Training"), which matches their RMS to AdamW's:
learning rates and weight decay tuned for AdamW carry over unchanged.
"""

from __future__ import annotations

import math
from typing import Iterable

import torch
import torch.nn as nn

NEWTON_SCHULZ = (3.4445, -4.7750, 2.0315)


def newton_schulz(matrix: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """Approximate the orthogonal factor ``U V^T`` of ``matrix = U S V^T``; singular values end up in about [0.7, 1.2]."""
    a, b, c = NEWTON_SCHULZ
    native_bf16 = matrix.is_cuda and torch.cuda.get_device_capability(matrix.device)[0] >= 8
    x = matrix.to(torch.bfloat16 if native_bf16 else torch.float32)
    tall = x.size(-2) > x.size(-1)
    if tall:
        x = x.mT
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        gram = x @ x.mT
        x = a * x + (b * gram + c * gram @ gram) @ x
    return (x.mT if tall else x).to(matrix.dtype)


def muon_parameter_groups(model: nn.Module, adamw_modules: Iterable[str] = ("head", "lm_head")) -> list[dict]:
    """Split ``model``'s parameters into a Muon group and an AdamW group.

    AdamW gets every parameter with fewer than two dimensions, every
    ``nn.Embedding`` weight (with any output layer tied to it) and every
    parameter of a module named in ``adamw_modules``, matched against the
    qualified name or its last component. Muon gets the remaining matrices.
    """
    names = set(adamw_modules)
    adamw = {
        id(p)
        for name, module in model.named_modules()
        if isinstance(module, nn.Embedding) or name in names or name.rsplit(".", 1)[-1] in names
        for p in module.parameters(recurse=False)
    }
    muon, other = [], []
    for p in model.parameters():  # tied weights appear once
        (other if p.ndim < 2 or id(p) in adamw else muon).append(p)
    return [{"params": muon, "use_muon": True}, {"params": other, "use_muon": False}]


class Muon(torch.optim.Optimizer):
    """Muon for hidden weight matrices, AdamW for every other parameter.

    ``params`` holds tensors, split by shape (two or more dimensions go to
    Muon), or groups with a ``use_muon`` flag such as
    :func:`muon_parameter_groups` returns. ``lr`` and decoupled
    ``weight_decay`` apply to both kinds; ``momentum``, ``nesterov`` and
    ``ns_steps`` to Muon; ``betas`` and ``eps`` to AdamW.
    """

    def __init__(
        self,
        params,
        lr: float = 3e-4,
        weight_decay: float = 0.0,
        momentum: float = 0.95,
        nesterov: bool = True,
        ns_steps: int = 5,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
    ):
        params = list(params)
        if params and not isinstance(params[0], dict):
            params = [
                {"params": [p for p in params if p.ndim >= 2], "use_muon": True},
                {"params": [p for p in params if p.ndim < 2], "use_muon": False},
            ]
        groups = [{**group, "params": list(group["params"])} for group in params]
        if any("use_muon" not in group for group in groups):
            raise ValueError("every parameter group needs a 'use_muon' flag")
        defaults = dict(
            lr=lr, weight_decay=weight_decay, momentum=momentum, nesterov=nesterov,
            ns_steps=ns_steps, betas=betas, eps=eps,
        )
        super().__init__([group for group in groups if group["params"]], defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            update = self._muon_update if group["use_muon"] else self._adamw_update
            for p in group["params"]:
                if p.grad is None:
                    continue
                direction = update(p.grad, self.state[p], group)
                p.mul_(1 - group["lr"] * group["weight_decay"])
                p.add_(direction, alpha=-group["lr"])
        return loss

    @staticmethod
    def _muon_update(grad, state, group):
        if not state:
            state["momentum_buffer"] = torch.zeros_like(grad)
        buffer = state["momentum_buffer"]
        buffer.lerp_(grad, 1 - group["momentum"])
        momentum = grad.lerp(buffer, group["momentum"]) if group["nesterov"] else buffer
        matrix = newton_schulz(momentum.reshape(len(momentum), -1), group["ns_steps"])
        return matrix.reshape_as(grad) * (0.2 * math.sqrt(max(matrix.shape)))

    @staticmethod
    def _adamw_update(grad, state, group):
        beta1, beta2 = group["betas"]
        if not state:
            state["step"] = 0
            state["exp_avg"] = torch.zeros_like(grad)
            state["exp_avg_sq"] = torch.zeros_like(grad)
        state["step"] += 1
        state["exp_avg"].lerp_(grad, 1 - beta1)
        state["exp_avg_sq"].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        corrected = (state["exp_avg_sq"] / (1 - beta2 ** state["step"])).sqrt().add_(group["eps"])
        return state["exp_avg"] / (1 - beta1 ** state["step"]) / corrected
'''  # laboratory/optimization/muon.py
_ns = {"__name__": "muon"}
exec(compile(MUON_SRC, "muon.py", "exec"), _ns)
Muon = _ns["Muon"]


def download(url, path, size=None):
    """Resumable download; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


t0 = time.time()
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "div2k": ("https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip", 448_993_893),
}
with ThreadPoolExecutor(2) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
IMAGES = []   # (name, bucket, clean uint8 HWC)
with zipfile.ZipFile(P["div2k"]) as z:
    for bucket, (h, w, ids) in SIZES.items():
        for i in ids:
            im = np.asarray(Image.open(io.BytesIO(z.read(f"DIV2K_valid_HR/{i:04d}.png"))).convert("RGB"))
            top, left = (im.shape[0] - h) // 2, (im.shape[1] - w) // 2
            IMAGES.append((f"{i:04d}", bucket, np.ascontiguousarray(im[top:top + h, left:left + w])))
TIMES["downloads"] = time.time() - t0
print([(n, b, im.shape) for n, b, im in IMAGES], f"downloads {TIMES['downloads'] / 60:.1f} min")


def noisy_of(k):
    """DMID's protocol (sample std exactly σ/255, unclipped); seeds outside every test-set range."""
    x = torch.from_numpy(IMAGES[k][2]).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(9_000_000 + 1000 * SIGMA + k))
    return x + (SIGMA / 255) * n / n.std(unbiased=False)


def psnr(out, clean_u8):
    a = (out[0].clamp(0, 1) * 255).round().double()
    b = torch.from_numpy(clean_u8).permute(2, 0, 1).to(a.device).double()
    return 10 * math.log10(255 ** 2 / (a - b).pow(2).mean().item())

## 1 · ADM (fp32 weights, fp16 autocast, gradient checkpointing) with per-conv vector and scalar gains

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def checkpoint(func, inputs, params, flag):
    if flag and torch.is_grad_enabled():
        return torch.utils.checkpoint.checkpoint(func, *inputs, use_reentrant=False)
    return func(*inputs)


def sdpa_attention(self, qkv):
    """ADM's legacy attention through SDPA with 4-D (batch, heads, tokens, channels) inputs, so the fused kernels apply."""
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.reshape(bs, self.n_heads, ch, length).transpose(2, 3).contiguous()
               for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(2, 3).reshape(bs, -1, length)


gd_unet.checkpoint = checkpoint
gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)


def make_ctx(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=False,
                     use_scale_shift_norm=True, use_checkpoint=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False)
    decoder = {id(mod) for part in (m.output_blocks, m.out) for mod in part.modules()}
    gains, scalars, decoder_gains = [], [], []
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            g = nn.Parameter(torch.ones(mod.out_channels, device=device))
            s = nn.Parameter(torch.ones(1, device=device))
            mod.register_forward_hook(lambda module, inp, out, g=g, s=s: out * (g * s).view(1, -1, 1, 1).to(out.dtype))
            gains.append(g)
            scalars.append(s)
            if id(mod) in decoder:
                decoder_gains.append(g)
    norms = [p for mod in m.modules() if isinstance(mod, nn.GroupNorm) for p in (mod.weight, mod.bias)]
    subsets = {"full": gains + norms, "norm": norms, "decoder": decoder_gains, "scalar": scalars}
    return dict(dev=device, adm=m, gains=gains, scalars=scalars, norms=norms, norms0=[p.detach().clone() for p in norms],
                subsets=subsets)


CTX = [make_ctx(d) for d in DEVICES]
ADM_STATE.clear()
print({k: sum(p.numel() for p in v) for k, v in CTX[0]["subsets"].items()}, "parameters per subset")


@torch.no_grad()
def reset(ctx):
    for p in ctx["gains"] + ctx["scalars"]:
        p.fill_(1.0)
    for p, v in zip(ctx["norms"], ctx["norms0"]):
        p.copy_(v)
    for p in ctx["gains"] + ctx["scalars"] + ctx["norms"]:
        p.requires_grad_(False)

## 2 · One-step posterior mean, scoring and R2R tuning of one subset

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()


def matched(sigma01):
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


def tweedie(model, y, sigma01):
    t, ab = matched(sigma01)
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = math.sqrt(ab) * (2 * yp - 1)
    with torch.autocast("cuda", dtype=torch.float16):
        eps = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3]
    return (((x - math.sqrt(1 - ab) * eps.float()) / math.sqrt(ab) + 1) / 2)[..., :h, :w]


@torch.no_grad()
def score(ctx, k):
    y = noisy_of(k)[None].to(ctx["dev"])
    return psnr(tweedie(ctx["adm"], y, SIGMA / 255), IMAGES[k][2])


def crops(img, gen, n=BATCH, size=CROP):
    _, H, W = img.shape
    ii = torch.randint(0, H - size + 1, (n,), generator=gen).tolist()
    jj = torch.randint(0, W - size + 1, (n,), generator=gen).tolist()
    return torch.stack([img[:, i:i + size, j:j + size] for i, j in zip(ii, jj)])


def tune(ctx, k, subset):
    dev, s = ctx["dev"], SIGMA / 255
    reset(ctx)
    params = ctx["subsets"][subset]
    for p in params:
        p.requires_grad_(True)
    noisy = noisy_of(k).to(dev)
    opt = Muon(params, lr=LR, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda")
    gen, ngen = torch.Generator().manual_seed(k), torch.Generator(device=dev).manual_seed(77_000 + k)
    for _ in range(STEPS):
        c = crops(noisy, gen)
        z = torch.randn(c.shape, generator=ngen, device=dev)
        loss = F.mse_loss(tweedie(ctx["adm"], c + ALPHA * s * z, s * math.sqrt(1 + ALPHA ** 2)), c - s * z / ALPHA)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    out = score(ctx, k)
    reset(ctx)
    return out

## 3 · Every (image, subset)

In [ ]:
def run_job(ctx, job):
    k, subset = job
    t0 = time.time()
    return dict(image=IMAGES[k][0], bucket=IMAGES[k][1], k=k, subset=subset, psnr=tune(ctx, k, subset), seconds=time.time() - t0)


def worker_pool(jobs, fn):
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while True:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))
                torch.cuda.empty_cache()

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    while any(th.is_alive() for th in threads):
        time.sleep(10)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
for c in CTX:
    reset(c)
BASE = {k: score(CTX[k % len(CTX)], k) for k in range(len(IMAGES))}
res, fail = worker_pool([(k, s) for s in SUBSETS for k in range(len(IMAGES))], run_job)
TIMES["tuning"] = time.time() - t0
ROWS = pd.DataFrame(res)
ROWS["base"] = ROWS.k.map(BASE)
ROWS["gain"] = ROWS.psnr - ROWS.base
ROWS.to_csv(f"{OUT}/gains.csv", index=False)
print(f"{len(res)} runs, {len(fail)} failed, {TIMES['tuning'] / 60:.1f} min")

## 4 · Pre-registered selection

In [ ]:
table = ROWS.pivot_table(index="subset", columns="bucket", values="gain", aggfunc="mean")
table["mean"] = ROWS.groupby("subset").gain.mean()
table["better"] = ROWS.groupby("subset").gain.apply(lambda g: int((g > 0).sum()))
print(table.round(4).to_string())
eligible = table[(table["768x512"] > 0) & (table["512x512"] > 0)]
chosen = str(eligible["mean"].idxmax()) if len(eligible) else None
print("\nchosen subset:", chosen)
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, table=table.reset_index().to_dict("records"), chosen=chosen,
                   failed=[j for j, _ in fail]), f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))